<a href="https://colab.research.google.com/github/njones61/xslope/blob/main/notebooks/xslope_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XSLOPE - Slope Design

This notebook finds the critical slope angle that produces a target design level using the limit equilibrium method. It sweeps a range of slope angles, evaluates a design metric for each, and interpolates to find the angle that meets the target. Two design modes are supported:

- **Factor of safety** (`design_mode = "fs"`) — find the angle that yields a target factor of safety.
- **Reliability** (`design_mode = "reliability"`) — find the angle that yields a target reliability $R = P(FS > 1)$. This requires material standard deviations (columns L–Q of the **mat** sheet).

> **Note:** This design workflow supports **profile-line inputs only.** It redesigns the slope by editing profile-line vertices and rebuilding the material polygons and ground surface, which has no direct equivalent for polygon-based geometry inputs. Use a profile-line input file with this notebook.

## Install xslope and import functions

In [ ]:
%%capture
!pip install xslope

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

from xslope.fileio import load_slope_data, build_ground_surface_from_polygons
from xslope.mesh import build_polygons
from xslope.plot import plot_solution, plot_reliability_results, plot_inputs
from xslope.search import circular_search, noncircular_search
from xslope.advanced import reliability as reliability_analysis

## Upload Excel Template

In [ ]:
from google.colab import files
upload = files.upload()
file_name = list(upload.keys())[0]

# See if uploaded file is a zip archive. If so, unzip it
if file_name.endswith('.zip'):
  import zipfile
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
    zip_ref.extractall()
    extracted_files = zip_ref.namelist()

    excel_file_found = False
    for f in extracted_files:
      if f.endswith('.xlsx'):
        file_name = f
        excel_file_found = True
        print(f"Found Excel file: {file_name}")
        break

    if not excel_file_found:
        print("Error: No .xlsx file found in the uploaded archive.")
        file_name = None

## Load slope data

In [ ]:
slope_data = load_slope_data(file_name)
plot_inputs(slope_data, mode='lem', save_png=False)

## Design Parameters

Choose the design mode and target. In `fs` mode the target is `design_fs`; in `reliability` mode the target is `design_reliability`. The target must fall within the range produced by the sweep — if it does not, widen `beta1`/`beta2` or change the target. `toe_index` is the index of the toe point on the first profile line and `slope_index` is the point at the top of the slope face (the one that is moved to change the angle).

In [ ]:
# @title Select Options {"run":"auto"}
method = "bishop" # @param ["oms","bishop","janbu","corps_engineers","lowe_karafiath","spencer","morgenstern_price"]
num_slices = 30 # @param {"type":"integer"}
design_mode = "reliability" # @param ["fs","reliability"]
surface_type = "circular" # @param ["circular","non_circular"]

beta1 = 20  # @param {"type":"number"}
beta2 = 30  # @param {"type":"number"}
num_angles = 10  # @param {"type":"integer"}
toe_index = 1  # @param {"type":"integer"}
slope_index = 2  # @param {"type":"integer"}

design_fs = 1.2  # @param {"type":"number"}
design_reliability = 0.75  # @param {"type":"number"}

save_png = True # @param {"type":"boolean"}

## Helper functions

`rebuild_geometry` regenerates the material polygons and ground surface after a profile point is moved — slice weights are computed from the polygons, so they must be rebuilt or they stay pinned to the original geometry. `evaluate` returns the design metric (FS or reliability) for the current geometry, and `set_slope_angle` moves the slope point to a given angle. Bishop's method is used by default: for the simple circular surfaces here it gives the same factor of safety as Spencer but stays fast, which matters because reliability mode runs several searches per angle.

In [ ]:
def rebuild_geometry(slope_data):
    """Rebuild the polygon geometry after editing profile_lines. Slice weights,
    layer heights, and base materials all come from slope_data['polygons'], so
    they must be regenerated (along with the ground surface) after each edit."""
    polys = [
        {'polygon': Polygon(p['coords']), 'mat_id': p['mat_id']}
        for p in build_polygons(slope_data={'profile_lines': slope_data['profile_lines'],
                                             'max_depth': slope_data.get('max_depth')})
    ]
    slope_data['polygons'] = polys
    ground_surface, domain_polygon = build_ground_surface_from_polygons(polys)
    slope_data['ground_surface'] = ground_surface
    slope_data['domain_polygon'] = domain_polygon


def run_search(slope_data):
    """Run the search for the selected surface_type; returns the sorted fs_cache."""
    if surface_type == "circular":
        fs_cache, converged, search_path, _ = circular_search(
            slope_data, method, num_slices=num_slices)
    else:
        fs_cache, converged, search_path = noncircular_search(
            slope_data, method, num_slices=num_slices)
    return fs_cache, converged, search_path


def evaluate(slope_data):
    """Return (metric, payload) for the current geometry per design_mode:
      fs          -> metric = critical FS,        payload = critical fs_cache entry
      reliability -> metric = reliability P(FS>1), payload = reliability result dict"""
    if design_mode == "fs":
        fs_cache, _, _ = run_search(slope_data)
        return fs_cache[0]['FS'], fs_cache[0]
    else:
        circular = (surface_type == "circular")
        success, result = reliability_analysis(
            slope_data, method, circular=circular, debug_level=0)
        if not success:
            raise RuntimeError(f"Reliability analysis failed: {result}")
        return result['reliability'], result


def set_slope_angle(slope_data, beta_deg):
    """Move the slope-top point so the face makes angle beta_deg, then rebuild."""
    profile = slope_data['profile_lines'][0]['coords']
    x_toe, y_toe = profile[toe_index]
    _x_top, y_top = profile[slope_index]
    # tan(beta) = (y_top - y_toe) / (x_top - x_toe)
    x_top_new = x_toe + (y_top - y_toe) / math.tan(math.radians(beta_deg))
    slope_data['profile_lines'][0]['coords'][slope_index] = (x_top_new, y_top)
    rebuild_geometry(slope_data)
    return x_top_new, y_top


# Mode-specific configuration: target value, labels, and output filename.
if design_mode == "fs":
    target = design_fs
    metric_label = "Factor of Safety"
    metric_short = "FS"
    out_png = "fs_vs_slope_angle.png"
elif design_mode == "reliability":
    target = design_reliability
    metric_label = "Reliability,  P(FS > 1)"
    metric_short = "R"
    out_png = "reliability_vs_slope_angle.png"
else:
    raise ValueError(f"Unknown design_mode: {design_mode!r} (use 'fs' or 'reliability')")

## Sweep slope angles

In [ ]:
betas = np.linspace(beta1, beta2, num=num_angles)
metric_results = np.zeros_like(betas)
payloads = [None] * len(betas)
for i, beta in enumerate(betas):
    x_top_new, y_top = set_slope_angle(slope_data, beta)
    metric_results[i], payloads[i] = evaluate(slope_data)
    if design_mode == "reliability":
        pf = payloads[i]['prob_failure']
        print(f"Slope angle: {beta:.1f}\u00b0, slope point x={x_top_new:.2f}  "
              f"R={metric_results[i]*100:.2f}%, Pf={pf*100:.2f}%")
    else:
        print(f"Slope angle: {beta:.1f}\u00b0, slope point x={x_top_new:.2f}  "
              f"FS={metric_results[i]:.3f}")

print("\nBeta sweep finished. Results:")
for beta, m in zip(betas, metric_results):
    print(f"  \u03b2_slope = {beta:.1f}\u00b0: {metric_short} = {m:.3f}")

## Interpolate critical slope angle

In [ ]:
# Both metrics decrease monotonically as the slope steepens, so interpolate the
# target with the arrays reversed (np.interp needs an ascending x table).
m_min, m_max = metric_results.min(), metric_results.max()
if not (m_min <= target <= m_max):
    print(f"Target {metric_short}={target} is outside the computed range "
          f"[{m_min:.3f}, {m_max:.3f}].")
    print(f"Adjust beta1/beta2 so the range brackets {metric_short}={target} and re-run.")
else:
    critical_slope_angle = float(np.interp(target, metric_results[::-1], betas[::-1]))
    print(f"Critical slope angle for {metric_short}={target}: {critical_slope_angle:.2f}\u00b0")

## Verify at critical slope angle

In [ ]:
# Re-run at the interpolated angle to confirm the metric is close to the target.
set_slope_angle(slope_data, critical_slope_angle)
metric_at_critical, payload = evaluate(slope_data)
print(f"{metric_short} at critical slope angle ({critical_slope_angle:.2f}\u00b0): "
      f"{metric_at_critical:.3f}")

if design_mode == "fs":
    plot_solution(slope_data, payload['slices'], payload['failure_surface'],
                  payload['solver_result'], save_png=save_png)
else:
    print(f"Probability of failure at critical angle: {payload['prob_failure']*100:.2f}%")
    plot_reliability_results(slope_data, payload, save_png=save_png)

## Plot design metric vs slope angle

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(betas, metric_results, marker='o')

# Horizontal line at the design target
ax.axhline(y=target, color='r', linestyle='--', linewidth=0.8,
           label=f'{metric_short} = {target}')

# Vertical line at critical slope angle
ax.axvline(x=critical_slope_angle, color='gray', linestyle='--', linewidth=0.8,
           label=f'\u03b2 = {critical_slope_angle:.1f}\u00b0')

# Mark the intersection point
ax.plot(critical_slope_angle, metric_at_critical, 's', color='r', markersize=8, zorder=5)

ax.set_xlabel('Slope Angle (degrees)')
ax.set_ylabel(metric_label)
ax.set_title(f'{metric_label} vs Slope Angle')
ax.legend()
ax.grid()
plt.tight_layout()
if save_png:
    plt.savefig(out_png, dpi=600)
plt.show()